# Synonym RxNorm Acquisition

## Check Python Version

In [1]:
!python --version

Python 3.11.9


## Setup Config

In [2]:
DATASET_DIR = "../../datasets/"

import pathlib
OUTPUT_DIR = pathlib.Path(DATASET_DIR)
print(f"Dataset Directory: {OUTPUT_DIR}")

FILE_UNIQUE_DRUG_DDI = "unique_drug_interactions.csv"
UNIQUE_DRUG_DDI_PATH = OUTPUT_DIR.joinpath(FILE_UNIQUE_DRUG_DDI)
print(f"Unique Drug Dataset Location: {UNIQUE_DRUG_DDI_PATH}")

FILE_FOUND_SYNONYM_DRUG_DDI = "resolved_drug_interactions_synonyms.csv"
RESOLVED_SYNONYM_PATH = OUTPUT_DIR.joinpath(FILE_FOUND_SYNONYM_DRUG_DDI)
print(f"Resolved Drug Synonyms from Drug Interactions Location: {RESOLVED_SYNONYM_PATH}")

FILE_NOTFOUND_RXCUI_DRUG_DDI = "unresolved_drug_interactions_rxcui.csv"
UNRESOLVED_RXCUI_PATH = OUTPUT_DIR.joinpath(FILE_NOTFOUND_RXCUI_DRUG_DDI)
print(f"Unresolved Drug RxCUI from Drug Interactions Location: {UNRESOLVED_RXCUI_PATH}")

FILE_NOTFOUND_SYNONYM_DRUG_DDI = "unresolved_drug_interactions_synonyms.csv"
UNRESOLVED_SYNONYM_PATH = OUTPUT_DIR.joinpath(FILE_NOTFOUND_SYNONYM_DRUG_DDI)
print(f"Unresolved Drug Synonyms from Drug Interactions Location: {UNRESOLVED_SYNONYM_PATH}")

Dataset Directory: ..\..\datasets
Unique Drug Dataset Location: ..\..\datasets\unique_drug_interactions.csv
Resolved Drug Synonyms from Drug Interactions Location: ..\..\datasets\resolved_drug_interactions_synonyms.csv
Unresolved Drug RxCUI from Drug Interactions Location: ..\..\datasets\unresolved_drug_interactions_rxcui.csv
Unresolved Drug Synonyms from Drug Interactions Location: ..\..\datasets\unresolved_drug_interactions_synonyms.csv


## Read Unique Drug from DDI Dataset

In [3]:
import requests
import pandas as pd
from tqdm import tqdm

df = pd.read_csv(UNIQUE_DRUG_DDI_PATH)
print(f"Total Drug: {df.shape[0]}")

Total Drug: 1701


## Synonym Resolver using RxNorm

In [4]:
import time
import requests

class RxNormSynonymResolver:
    ALLOWED_TYPES = {"IN", "SY", "FSY"}
    BASE_URL = "https://rxnav.nlm.nih.gov/REST"

    def _get_rxcui(self, drug_name: str):
        try:
            url = f"{self.BASE_URL}/rxcui.json"
    
            response = requests.get(
                url,
                params={"name": drug_name},
                timeout=10
            )
    
            response.raise_for_status()
            data = response.json()
    
            rxnorm_ids = (
                data.get("idGroup", {})
                .get("rxnormId", [])
            )
    
            if not rxnorm_ids:
                return None
    
            return rxnorm_ids[0]
    
        except Exception as e:
            print(f"[RXCUI ERROR] {drug_name}: {e}")
            return None


    def _get_aliases(self, name: str, rxcui: str):
        try:
            url = f"{self.BASE_URL}/rxcui/{rxcui}/proprietary.json"
    
            response = requests.get(url, timeout=10)
            response.raise_for_status()
    
            data = response.json()
    
            proprietary_info = (
                data.get("proprietaryGroup", {})
                .get("proprietaryInfo", [])
            )
    
            aliases = set()
            for item in proprietary_info:
                if item.get("type") not in self.ALLOWED_TYPES:
                    continue
                    
                proprietary_name = item.get("name")
    
                if proprietary_name and name.lower() != proprietary_name.lower():
                    aliases.add(proprietary_name.lower())
    
            return {
                "rx_cui": rxcui,
                "name": name,
                "aliases": sorted(list(aliases))
            }
    
        except Exception as e:
            print(f"[ALIAS ERROR] {rxcui}: {e}")
    
            return None

    def get_synonyms(self, name):
        drug = str(name).strip().lower()
    
        rxcui = self._get_rxcui(drug)
        if not rxcui:
            return {
                "status": "unknown_rxcui",
                "data": {
                    "raw_name": drug
                }
            }
    
        alias_data = self._get_aliases(drug, rxcui)
        if not alias_data:
            return {
                "status": "unknown_aliases",
                "data": {
                    "raw_name": drug
                }
            }
            
        return {
            "status": "ok",
            "data": {
                "raw_name": drug, 
                "rxcui": rxcui,
                "aliases": alias_data["aliases"]
            }
        }

## Resolve Synonyms for All Unique Drugs in the DDI Dataset

In [5]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm


results = []
unknown_rxcui = []
unknown_aliases = []

# Maksimal 5 request diproses secara paralel
resolver = RxNormSynonymResolver()

with ThreadPoolExecutor(max_workers=5) as executor:

    futures = {
        executor.submit(resolver.get_synonyms, drug): drug
        for drug in df["drug"]
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures)
    ):
        try:
            result = future.result()

            if result["status"] == "ok":
                results.append(result["data"])

            elif result["status"] == "unknown_rxcui":
                unknown_rxcui.append(result["data"])

            elif result["status"] == "unknown_aliases":
                unknown_aliases.append(result["data"])

        except Exception as e:
            print(f"[ERROR] {futures[future]}: {e}")

100%|██████████| 1701/1701 [20:10<00:00,  1.41it/s]


In [6]:
# Sort lagi karena parallel processing
# membuat urutan hasil tidak terjamin
results.sort(key=lambda x: x["raw_name"])
unknown_rxcui.sort(key=lambda x: x["raw_name"])
unknown_aliases.sort(key=lambda x: x["raw_name"])

In [7]:
# Export result
df_result = pd.DataFrame(results)
df_result.to_csv(RESOLVED_SYNONYM_PATH, index=False)

# Export unknown RxCUI
df_unknown_rx_cui = pd.DataFrame(unknown_rxcui)
df_unknown_rx_cui.to_csv(UNRESOLVED_RXCUI_PATH, index=False)


# Export unknown aliases
df_unknown_aliases = pd.DataFrame(unknown_aliases)
df_unknown_aliases.to_csv(UNRESOLVED_SYNONYM_PATH, index=False)

print(f"Success         : {len(results)}")
print(f"Unknown RxCUI   : {len(unknown_rxcui)}")
print(f"Unknown aliases : {len(unknown_aliases)}")

Success         : 1614
Unknown RxCUI   : 87
Unknown aliases : 0
